##L03: Bronze load: land raw taxi data into Delta

###Step 1. Read each monthly file and add lineage columns
Reading file by file avoids Parquet type clashes between months. source_file records where each row came from; ingested_at records when.

In [0]:
%python
from pyspark.sql import functions as F

base = "/Volumes/da/bronze/landing/yellow_trips"
files = [f.path for f in dbutils.fs.ls(base) if f.name.endswith(".parquet")]
print(files)

frames = []
for p in sorted(files):
    d = (spark.read.parquet(p)
         .withColumn("source_file", F.lit(p.split("/")[-1]))
         .withColumn("ingested_at", F.current_timestamp()))
    print(p.split("/")[-1], d.count())
    frames.append(d)

bronze = frames[0]
for d in frames[1:]:
    bronze = bronze.unionByName(d, allowMissingColumns=True)

###Step 2. Write the bronze table
Bronze keeps the data exactly as received; do not clean anything here.

In [0]:
(bronze.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable("da.bronze.yellow_trips"))

###SQL
Alternatatively we can use SQL instead of above two code blocks.

In [0]:
%sql
DROP TABLE IF EXISTS da.bronze.yellow_trips;
CREATE TABLE da.bronze.yellow_trips
AS
SELECT 
  *,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingested_at
FROM parquet.`/Volumes/da/bronze/landing/yellow_trips/*.parquet`
;

SELECT * FROM da.bronze.yellow_trips LIMIT 10;

###Step 3. Load the zone lookup

In [0]:
zones = spark.read.csv("/Volumes/da/bronze/landing/reference/taxi_zone_lookup.csv",
                       header=True, inferSchema=True)
zones.write.mode("overwrite").saveAsTable("da.bronze.taxi_zones")
display(zones.limit(5))

Also in SQL

In [0]:
%sql
CREATE OR REPLACE TABLE da.bronze.taxi_zones AS
SELECT * FROM read_files(
  '/Volumes/da/bronze/landing/reference/taxi_zone_lookup.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

SELECT * FROM da.bronze.taxi_zones LIMIT 5;

###Step 4. Reconcile counts
The table row count must equal the sum of the file counts printed in step 1. Check which columns exist and how many rows each month has.

In [0]:
%sql
SELECT 
    source_file, 
    COUNT(*) AS rows,
    MIN(tpep_pickup_datetime) AS first_pickup, 
    MAX(tpep_pickup_datetime) AS last_pickup

FROM da.bronze.yellow_trips 
GROUP BY source_file 
ORDER BY source_file
;

-- DESCRIBE da.bronze.yellow_trips;